In [5]:
TEST_VERSION = 'v2'       
DATA_ROOT = None          
TEST_ROOT = None         
OUTPUT_DIR = '/kaggle/working/airi_v4_fulltrain' if __import__('pathlib').Path('/kaggle/working').exists() else '/mnt/data/_airi_v4_fulltrain'
BASE_EPOCHS = 30
TEACHER_EPOCHS = 60
PROXY_TREES = 250
BATCH_SIZE = 16
REUSE_CHECKPOINTS = False
RUN_FINAL_INFERENCE = True
SMOKE_TEST = False
assert TEST_VERSION in ('v1','v2')
assert BASE_EPOCHS == 30 and TEACHER_EPOCHS == 60 and PROXY_TREES == 250
print('TEST_VERSION:', TEST_VERSION, '| refit epochs:', BASE_EPOCHS, TEACHER_EPOCHS)

TEST_VERSION: v2 | refit epochs: 30 60


## 1. Среда и воспроизводимость

In [6]:
import sys, os, csv, re, json, hashlib, random, copy, tempfile, gc, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import uniform_filter1d
from scipy.signal import welch, butter, sosfiltfilt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from catboost import CatBoostClassifier
import joblib

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, max(1, os.cpu_count() or 1)))
OUT = Path(OUTPUT_DIR).expanduser().resolve()
WEIGHTS = OUT/'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
ACTIONS = ('Close','Fist','Index','Little','Middle','Open','Ring','Thumb')
CONTEXT_SAMPLES = 1000
CORE = 128
CONTEXT = 64
FS = 500
STRIDE = 20
WINDOWS_MS = (50,200,600)
OFFLINE_LAGS_MS = (-300,0,300)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_all(42)
print('PyTorch',torch.__version__,'Device',DEVICE,'Output',OUT)

PyTorch 2.11.0+cu130 Device cuda Output /kaggle/working/airi_v4_fulltrain


## 2. Строгий выбор train и теста (проверка V2 до обучения)

In [7]:
DATA_CANDIDATES = []
if DATA_ROOT is not None: DATA_CANDIDATES.append(Path(DATA_ROOT))
DATA_CANDIDATES += [Path('/kaggle/input/datasets/yalmawka/nauchnaya-orbita'), Path('/kaggle/input/nauchnaya-orbita')]
if Path('/kaggle/input').exists():
    DATA_CANDIDATES += [p.parent.parent for p in Path('/kaggle/input').rglob('train/metadata.csv')]
if SMOKE_TEST: DATA_CANDIDATES.insert(0,Path('/mnt/data/_airi_v4_fixture'))
DATA_CANDIDATES = list(dict.fromkeys(p.resolve() for p in DATA_CANDIDATES))
DATA_DIR = next((p for p in DATA_CANDIDATES if (p/'train/metadata.csv').is_file() and (p/'train/signals').is_dir() and (p/'train/targets').is_dir()), None)
if DATA_DIR is None: raise FileNotFoundError('Подключите nauchnaya-orbita с train/metadata.csv, train/signals и train/targets')

TEMPLATE_NAME = 'phase2_sample_submission.csv' if TEST_VERSION=='v2' else 'phase1_sample_submission.csv'
TEST_DIR_NAMES = ('phase2_test/signals','phase2_test','test/signals','test','phase2/signals') if TEST_VERSION=='v2' else ('phase1_test/signals','phase1_test')
SEARCH_ROOTS=[DATA_DIR]
if TEST_ROOT is not None: SEARCH_ROOTS.insert(0,Path(TEST_ROOT))
if Path('/kaggle/input').is_dir():SEARCH_ROOTS.append(Path('/kaggle/input'))
if SMOKE_TEST:SEARCH_ROOTS.insert(0,Path('/mnt/data/_airi_v4_fixture'))
templates=[]
for root in SEARCH_ROOTS:
    if (root/TEMPLATE_NAME).is_file():templates.append(root/TEMPLATE_NAME)
    if root.is_dir():templates += list(root.rglob(TEMPLATE_NAME))
templates=list(dict.fromkeys(p.resolve() for p in templates))
if not templates:raise FileNotFoundError('Выбран '+TEST_VERSION+', но '+TEMPLATE_NAME+' не найден ни в одном Kaggle Input. Никакого V1 fallback.')

REQUIRED_COLUMNS=['row_id','recording_id','sample_index']+[f'angle_{i:02d}' for i in range(20)]
choices=[]
for template_candidate in templates:
    if template_candidate.name!=TEMPLATE_NAME:continue
    with template_candidate.open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=REQUIRED_COLUMNS:continue
        rec_ids=set();row_count=0
        for row in reader:
            rec_ids.add(row['recording_id']);row_count+=1
    if row_count==0:continue
    candidate_dirs=[]
    for root in [template_candidate.parent,*SEARCH_ROOTS]:
        for rel in TEST_DIR_NAMES:candidate_dirs.append(root/rel)
    if Path('/kaggle/input').is_dir():
        for directory in Path('/kaggle/input').rglob('signals'):
            if directory.parent.name in ('phase2_test','test','phase2') if TEST_VERSION=='v2' else directory.parent.name=='phase1_test':
                candidate_dirs.append(directory)
    candidate_dirs=list(dict.fromkeys(p.resolve() for p in candidate_dirs))
    matches=[]
    for p in candidate_dirs:
        if not p.is_dir() or ('phase1' in str(p).lower() and TEST_VERSION=='v2'):continue
        signals={f.stem for f in p.glob('rec_*.npy')}
        if signals and signals==rec_ids:matches.append(p)
    if matches:choices.append((template_candidate,rec_ids,row_count,matches))
if len(choices)!=1:
    raise RuntimeError(f'{TEST_VERSION}: ожидается один однозначный template+test dataset, найдено {len(choices)}. Укажи TEST_ROOT.')
TEMPLATE,template_record_ids,template_row_count,matching=choices[0]
if len(matching)!=1:raise RuntimeError('Несколько подходящих тестовых каталогов: '+str(matching)+'; укажи TEST_ROOT')
TEST_SIGNALS=matching[0]
assert ('phase2_' in TEMPLATE.name)==(TEST_VERSION=='v2'),'Смешаны тестовые фазы'
if TEST_VERSION=='v2':
    assert TEMPLATE.name=='phase2_sample_submission.csv' and 'phase1' not in str(TEST_SIGNALS).lower()
    assert len(template_record_ids)==32, f'Ожидалось ровно 32 записи V2, найдено {len(template_record_ids)}'
else:assert TEMPLATE.name=='phase1_sample_submission.csv'
assert not TEST_SIGNALS.is_relative_to(DATA_DIR/'train')
assert OUT != DATA_DIR and not OUT.is_relative_to(DATA_DIR), 'Вывод нельзя записывать в Input'
print('V2 ASSERT PASSED' if TEST_VERSION=='v2' else 'V1 ASSERT PASSED')
print('Dataset:',DATA_DIR,' | Test:',TEST_SIGNALS,' | Template:',TEMPLATE)
print('Test recordings:',len(template_record_ids),'Template rows:',template_row_count)

V2 ASSERT PASSED
Dataset: /kaggle/input/datasets/yalmawka/nauchnaya-orbita/phase1_public  | Test: /kaggle/input/datasets/yalmawka/nauchnaya-orbita/phase2_public/test  | Template: /kaggle/input/datasets/yalmawka/nauchnaya-orbita/phase2_public/phase2_sample_submission.csv
Test recordings: 32 Template rows: 624032


## 3. Архитектура

In [8]:
def offline_features(x):
    x=np.asarray(x,dtype=np.float32)
    ax=np.abs(x); dx=np.abs(np.diff(x,axis=0,prepend=x[:1]))
    parts=[]
    for ms in WINDOWS_MS:
        n=max(2,round(FS*ms/1000))
        parts.extend([
            uniform_filter1d(x,size=n,axis=0,mode='nearest'),
            uniform_filter1d(ax,size=n,axis=0,mode='nearest'),
            np.sqrt(np.maximum(uniform_filter1d(x*x,size=n,axis=0,mode='nearest'),0)),
            uniform_filter1d(dx,size=n,axis=0,mode='nearest'),
        ])
    full=np.concatenate(parts,axis=1)
    grid=np.arange(0,len(x),STRIDE,dtype=np.int32)
    feat=np.concatenate([full[np.clip(grid+round(FS*lag/1000),0,len(x)-1)] for lag in OFFLINE_LAGS_MS],axis=1)
    return grid,np.ascontiguousarray(feat,dtype=np.float32)


def pad_and_slice(x,start,length):
    left=max(0,-start);right=max(0,start+length-x.shape[-1])
    chunk=x[...,max(0,start):min(x.shape[-1],start+length)]
    if left or right:
        chunk=np.pad(chunk,[(0,0)]*(chunk.ndim-1)+[(left,right)],mode='edge')
    assert chunk.shape[-1]==length
    return np.ascontiguousarray(chunk,dtype=np.float32)


def upsample(pred_grid,grid,n_samples,causal=False):
    assert not causal
    pred=np.stack([np.interp(np.arange(n_samples),grid,pred_grid[:,j]) for j in range(20)],axis=1)
    pred=np.asarray(np.clip(pred,-np.pi,np.pi),dtype=np.float32)
    assert pred.shape==(n_samples,20)
    return pred


class TCNBlock(nn.Module):
    def __init__(self,ch,d):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(ch,ch,3,padding=d,dilation=d),nn.BatchNorm1d(ch),nn.GELU(),nn.Dropout(.1),
                               nn.Conv1d(ch,ch,3,padding=d,dilation=d),nn.BatchNorm1d(ch),nn.Dropout(.1))
    def forward(self,x):return F.gelu(x+self.net(x))


class RegressionTCN(nn.Module):
    def __init__(self,n_features=216):
        super().__init__();self.embed=nn.Conv1d(n_features,64,1)
        self.net=nn.Sequential(*(TCNBlock(64,2**i) for i in range(5)))
        self.out=nn.Conv1d(64,20,1)
    def forward(self,x):return self.out(self.net(self.embed(x)))


class BiGRURegressor(nn.Module):
    def __init__(self,n_features=216):
        super().__init__()
        self.front=nn.Sequential(nn.Conv1d(n_features,96,1),nn.GELU(),nn.Conv1d(96,96,5,padding=2),nn.GELU())
        self.gru=nn.GRU(96,64,num_layers=2,dropout=.1,bidirectional=True,batch_first=True)
        self.head=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.GELU(),nn.Linear(64,20))
    def forward(self,x):
        z=self.front(x).transpose(1,2);h,_=self.gru(z)
        return self.head(h).transpose(1,2)


class HybridTCNBiGRU(nn.Module):
    def __init__(self,n_features=216):
        super().__init__()
        self.embed=nn.Conv1d(n_features,64,1)
        self.net=nn.Sequential(*(TCNBlock(64,2**i) for i in range(5)))
        self.gru=nn.GRU(64,64,num_layers=2,dropout=.1,bidirectional=True,batch_first=True)
        self.norm=nn.LayerNorm(128)
        self.head=nn.Sequential(nn.Conv1d(192,96,1),nn.GELU(),nn.Dropout(.1),nn.Conv1d(96,20,1))
    def forward(self,x):
        z=self.net(self.embed(x));h,_=self.gru(z.transpose(1,2));h=self.norm(h).transpose(1,2)
        return self.head(torch.cat((z,h),dim=1))


class GroupHybrid(HybridTCNBiGRU):
    def __init__(self,n_features):
        super().__init__(n_features)
        def replace(module):
            for name,child in list(module.named_children()):
                if isinstance(child,nn.BatchNorm1d):setattr(module,name,nn.GroupNorm(8,child.num_features))
                else:replace(child)
        replace(self)

In [9]:
def extract(x):
    """Portable: a single arbitrary-length EMG input, no metadata or labels."""
    x=np.asarray(x,dtype='float32')*1e6
    if x.ndim!=2 or x.shape[1]!=6 or len(x)<1001:raise ValueError('EMG must be T x 6, T > 1000')
    grid=np.arange(0,len(x),STRIDE)
    x=x-np.median(x,axis=0)
    vals=[]
    for width in (25,100,300):
        av=uniform_filter1d(np.abs(x),size=width,axis=0,mode='nearest')
        rms=np.sqrt(np.maximum(uniform_filter1d(x*x,size=width,axis=0,mode='nearest'),0))
        wl=uniform_filter1d(np.abs(np.diff(x,axis=0,prepend=x[:1])),size=width,axis=0,mode='nearest')
        vals.extend((np.log1p(av),np.log1p(rms),np.log1p(wl)))
    env=np.concatenate(vals,axis=1)
    local=np.concatenate([env[np.clip(grid+lag,0,len(x)-1)] for lag in (-150,0,150)],axis=1)
    q10,q90=np.quantile(local,[.1,.9],axis=0)
    relative=(local-q10)/np.maximum(q90-q10,.15)
    features=np.concatenate((local,relative),axis=1).astype('float32')
    glob=np.concatenate([np.mean(local,0),np.std(local,0),*np.quantile(local,[.1,.5,.9],axis=0)]).astype('float32')
    return grid,features,glob


def global_features(x,r):
    x=np.asarray(x,dtype='float64')*1e6;x-=np.median(x,0)
    freq,psd=welch(x,fs=500,nperseg=256,axis=0)
    bands=[]
    for lo,hi in ((1,10),(10,20),(20,40),(40,60),(60,90),(90,130),(130,180),(180,250)):
        bands.append(np.log(np.maximum(psd[(freq>=lo)&(freq<hi)].mean(0),1e-12)))
    bands=np.array(bands)
    total=np.log(np.maximum(np.mean(x*x,0),1e-12))
    spatial=total-total.mean()
    c=np.corrcoef(x.T);corr=c[np.triu_indices(6,1)]
    local=r['f'][:,:162]
    q=np.quantile(local,[.05,.25,.5,.75,.95],axis=0)
    relative=q.reshape(5,-1,6);relative=relative-relative.mean(2,keepdims=True)
    return np.concatenate([r['g'],q.ravel(),relative.ravel(),bands.ravel(),(bands-bands.mean(1,keepdims=True)).ravel(),total,spatial,corr]).astype('float32')


def spectral_offline_features(x):
    grid,base=offline_features(x);parts=[]
    for lo,hi in ((20,60),(60,100),(100,180),(180,240)):
        sos=butter(3,[lo,hi],btype='bandpass',fs=500,output='sos')
        band=sosfiltfilt(sos,np.asarray(x,dtype='float64'),axis=0)
        for width in (100,300):
            env=np.sqrt(np.maximum(uniform_filter1d(band*band,size=width,axis=0,mode='nearest'),0))
            for lag in (-150,0,150):parts.append(env[np.clip(grid+lag,0,len(x)-1)])
    return grid,np.concatenate([base]+parts,axis=1).astype('float32')


def gesture_features(signal):
    grid,ft=offline_features(np.asarray(signal,dtype=np.float32))
    ft=ft[grid>=1000];chunks=[]
    for start in range(0,len(ft),50):
        seg=ft[start:start+50]
        if len(seg)<12:continue
        chunks.append(np.r_[seg.mean(0),seg.std(0)])
    if not chunks:chunks=[np.r_[ft.mean(0),ft.std(0)]]
    return np.asarray(chunks,dtype=np.float32)


def whole_record_features(signal):
    _,f,g=extract(signal)
    with np.errstate(invalid='ignore',divide='ignore'):
        result=global_features(signal,{'f':f,'g':g})
    return np.nan_to_num(result,nan=0.,posinf=0.,neginf=0.)

In [10]:
@torch.inference_mode()
def predict_regressor(net,signal,stats,prob,device='cpu',batch_size=1):
    x=np.asarray(signal,dtype=np.float32)
    grid,values=spectral_offline_features(x) if len(stats[0])==360 else offline_features(x)
    tau=getattr(net,'input_asinh_tau',None)
    if tau is not None:values=np.arcsinh(values/tau).astype(np.float32)
    mu,sd=stats
    values=np.clip((values-mu)/sd,-10,10).astype(np.float32)
    prob=np.asarray(prob,dtype=np.float32)
    values=np.concatenate((values,np.broadcast_to((prob-.125)*4,(len(values),8))),axis=1).T
    chunks=[pad_and_slice(values,start-64,256) for start in range(0,len(grid),128)]
    outputs=[];net.eval()
    for start in range(0,len(chunks),batch_size):
        xb=torch.from_numpy(np.stack(chunks[start:start+batch_size])).to(device)
        pred=net(xb)[:,:,64:192].transpose(1,2).cpu().numpy()
        outputs.extend(pred)
    pg=np.concatenate(outputs,axis=0)[:len(grid)]
    return upsample(pg,grid,len(x))

class AIRIModel:
    def __init__(self,weights=None,device=None,verify_hashes=True):
        self.root=Path(weights) if weights else Path(__file__).resolve().parent/'weights'
        self.device=torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        torch.set_num_threads(min(4,torch.get_num_threads()))
        self.manifest=json.loads((self.root/'manifest.json').read_text(encoding='utf-8'))
        if self.manifest.get('candidate')!='v4_asinh_fixed_epochs':
            raise ValueError('Inference code and weight release do not match')
        if verify_hashes:
            for name,expected in self.manifest['sha256'].items():
                actual=hashlib.sha256((self.root/name).read_bytes()).hexdigest()
                if actual!=expected:raise ValueError('Checkpoint checksum mismatch: '+name)
        makers={'tcn':RegressionTCN,'bigru':BiGRURegressor,'hybrid':HybridTCNBiGRU}
        self.regressors={}
        for arch in ('tcn','bigru','hybrid','teacher','teacher_spectral','teacher_spectral_seed27183','teacher_group_gain_day','teacher_asinh'):
            saved=torch.load(self.root/(arch+'.pt'),map_location='cpu',weights_only=False)
            model=(GroupHybrid(len(saved['stats'][0])+8) if arch=='teacher_group_gain_day' else makers['hybrid' if arch.startswith('teacher') else arch](len(saved['stats'][0])+8)).to(self.device)
            model.load_state_dict(saved['state_dict']);model.eval()
            if arch=='teacher_asinh':
                model.input_asinh_tau=np.asarray(saved['tau'],dtype=np.float32)
                if model.input_asinh_tau.shape!=(360,) or not np.isfinite(model.input_asinh_tau).all() or np.any(model.input_asinh_tau<=0):raise ValueError('Invalid asinh transform scales')
            self.regressors[arch]=(model,saved['stats'])
        self.proxy=CatBoostClassifier();self.proxy.load_model(str(self.root/'gesture_catboost.cbm'))
        self.classifiers=joblib.load(self.root/'whole_record_classifiers.joblib')

    def predict(self,signal):
        raw=np.asarray(signal)
        if raw.ndim!=2 or raw.shape[1]!=6 or len(raw)<=1000 or not np.isfinite(raw).all():
            raise ValueError('Expected finite EMG array (T,6), T > 1000')
        proxy=np.asarray(self.proxy.predict_proba(gesture_features(raw)),dtype=np.float64).mean(0)
        p=np.zeros(8,dtype=np.float32);p[np.asarray(self.proxy.classes_,dtype=int)]=proxy;p/=p.sum()
        gf=whole_record_features(raw)[None]
        teacher=np.mean([clf.predict_proba(gf)[0] for clf in self.classifiers],axis=0).astype(np.float32);teacher/=teacher.sum()
        predictions=[]
        for arch in ('tcn','bigru','hybrid'):
            net,stats=self.regressors[arch]
            predictions.append(predict_regressor(net,raw,stats,p,self.device))
        baseline=np.mean(predictions,axis=0)
        net,stats=self.regressors['teacher']
        continuous=predict_regressor(net,raw,stats,teacher,self.device)
        v1=(baseline+continuous)*.5
        net,stats=self.regressors['teacher_spectral']
        spectral=predict_regressor(net,raw,stats,teacher,self.device)
        net,stats=self.regressors['teacher_spectral_seed27183']
        spectral_b=predict_regressor(net,raw,stats,teacher,self.device)
        net,stats=self.regressors['teacher_group_gain_day']
        group=predict_regressor(net,raw,stats,teacher,self.device)
        result=((v1+(spectral+spectral_b)*.5+group)/3).astype(np.float32)
        net,stats=self.regressors['teacher_asinh']
        transformed=predict_regressor(net,raw,stats,teacher,self.device)
        result=((3*result+transformed)/4).astype(np.float32)
        if result.shape!=(len(raw),20) or not np.isfinite(result).all():raise RuntimeError('Invalid model output')
        return result


def write_submission(model,input_dir,template,output):
    input_dir=Path(input_dir).resolve();template=Path(template).resolve();output=Path(output).resolve()
    protected=[input_dir]
    for parent in input_dir.parents:
        if parent.name.lower()=='data' or (parent/'train/metadata.csv').is_file():
            protected.append(parent);break
    if any(output.is_relative_to(p) for p in protected):
        raise ValueError('Write submissions outside input data, including nested folders')
    if output==template or (output.exists() and template.exists() and os.path.samefile(output,template)):
        raise ValueError('Refusing to overwrite submission template')
    columns=['row_id','recording_id','sample_index']+[f'angle_{i:02d}' for i in range(20)]
    with template.open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=columns:raise ValueError('Unexpected template columns/order')
        rows=list(reader)
    if not rows:raise ValueError('Empty template')
    if any(None in r or any(r.get(k) is None for k in columns) or not r['row_id'] for r in rows):
        raise ValueError('Malformed template row')
    if len({r['row_id'] for r in rows})!=len(rows):raise ValueError('Duplicate row_id')
    by_record={}
    for row in rows:
        if re.fullmatch(r'rec_[0-9a-f]{16}',row['recording_id']) is None:raise ValueError('Invalid recording_id')
        if re.fullmatch(r'[0-9]+',row['sample_index']) is None:raise ValueError('Invalid sample_index')
        by_record.setdefault(row['recording_id'],[]).append(int(row['sample_index']))
    files={p.stem:p for p in input_dir.glob('*.npy')}
    if set(files)!=set(by_record):raise ValueError('Template and signal files must contain the same recordings')
    predictions={}
    for rid,indices in by_record.items():
        signal_path=files[rid].resolve()
        if not signal_path.is_relative_to(input_dir):raise ValueError('Recording path escapes input data')
        x=np.load(signal_path,allow_pickle=False)
        if x.ndim!=2 or x.shape[1]!=6 or len(x)<=1000 or not np.isfinite(x).all():raise ValueError('Invalid EMG input')
        if sorted(indices)!=list(range(1000,len(x))):raise ValueError('Missing/duplicate sample indices for '+rid)
        pred=np.asarray(model.predict(x))
        if pred.shape!=(len(x),20) or not np.isfinite(pred).all():raise ValueError('Invalid model prediction')
        predictions[rid]=pred
    output.parent.mkdir(parents=True,exist_ok=True)
    # A unique temporary file prevents interrupted writes from replacing a valid CSV.
    fd,temporary=tempfile.mkstemp(prefix=output.name+'.',suffix='.partial',dir=output.parent)
    try:
        with os.fdopen(fd,'w',encoding='utf-8',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=columns);writer.writeheader()
            for row in rows:
                pred=predictions[row['recording_id']][int(row['sample_index'])]
                writer.writerow(dict(zip(columns,[row['row_id'],row['recording_id'],row['sample_index']]+[format(float(v),'.9g') for v in pred])))
        os.replace(temporary,output)
    finally:
        if Path(temporary).exists():Path(temporary).unlink()
    return {'rows':len(rows),'recordings':len(by_record),'output':str(output)}

## 4. Загрузка всех 80 обучающих записей и контроль входов

In [11]:
meta = pd.read_csv(DATA_DIR/'train/metadata.csv')
need_meta={'recording_id','n_samples','action','day_id','session_id'}
assert need_meta.issubset(meta.columns), f'Не хватает колонок: {need_meta-set(meta.columns)}'
assert len(meta)==80 and meta.recording_id.is_unique
assert set(meta.action)==set(ACTIONS)
assert meta.day_id.nunique()==7 and meta.session_id.nunique()==10
assert all(set(g.action)==set(ACTIONS) and len(g)==8 for _,g in meta.groupby('session_id'))
records={}
for row in meta.itertuples(index=False):
    pathx=DATA_DIR/'train/signals'/f'{row.recording_id}.npy'
    pathy=DATA_DIR/'train/targets'/f'{row.recording_id}.npy'
    xx=np.load(pathx,allow_pickle=False).astype(np.float32)
    yy=np.load(pathy,allow_pickle=False).astype(np.float32)
    assert xx.shape==(row.n_samples,6) and yy.shape==(row.n_samples,20)
    assert len(xx)>1000 and np.isfinite(xx).all() and np.isfinite(yy).all()
    records[row.recording_id]={'x':xx,'y':yy,'day':row.day_id,'session':row.session_id,'action':row.action}
TRAIN_IDS=list(meta.recording_id)
DAYS=sorted(meta.day_id.unique())
assert set(TRAIN_IDS).isdisjoint(template_record_ids), 'Лик: train и test IDs пересекаются'
print('Train:',len(TRAIN_IDS),'Days:',DAYS,'Test:',len(template_record_ids))

h=hashlib.sha256(meta.sort_values('recording_id').to_csv(index=False).encode())
for rid in sorted(TRAIN_IDS):
    h.update(rid.encode());h.update(np.ascontiguousarray(records[rid]['x']).tobytes());h.update(np.ascontiguousarray(records[rid]['y']).tobytes())
DATA_FINGERPRINT=h.hexdigest()
TRAIN_CONFIG={'candidate':'v4_asinh_fixed_epochs','base_epochs':BASE_EPOCHS,'teacher_epochs':TEACHER_EPOCHS,
              'proxy_trees':PROXY_TREES,'batch_size':BATCH_SIZE,'dataset_sha256':DATA_FINGERPRINT,
              'source_version':'provided_airi_model_2026-11-22','base_seed':42,'teacher_seed':16180,
              'spectral_second_seed':27183}
TRAIN_SIGNATURE=hashlib.sha256(json.dumps(TRAIN_CONFIG,sort_keys=True).encode()).hexdigest()
print('Training signature:', TRAIN_SIGNATURE[:20])

Train: 80 Days: ['day_14fb2381', 'day_2e435bb4', 'day_6133f7ce', 'day_c7670e91', 'day_cda5287b', 'day_efee22d1', 'day_fad41c15'] Test: 32
Training signature: bcc24590e80d4c07746f


## 5. CatBoost: честные day-OOF вероятности для train + full-train classifier для test

In [12]:
SEGMENT_CACHE={rid:gesture_features(records[rid]['x']) for rid in TRAIN_IDS}

def make_catboost(ids, seed=42):
    xx=np.concatenate([SEGMENT_CACHE[r] for r in ids]).astype(np.float32)
    yy=np.concatenate([np.full(len(SEGMENT_CACHE[r]),ACTIONS.index(records[r]['action']),dtype=np.int32) for r in ids])
    assert set(np.unique(yy))==set(range(8)), 'Некоторые жесты отсутствуют в обучении'
    kwargs=dict(loss_function='MultiClass',iterations=PROXY_TREES,depth=5,learning_rate=.07,
                l2_leaf_reg=5.,random_seed=seed,task_type='GPU' if DEVICE.type=='cuda' else 'CPU',
                verbose=False,allow_writing_files=False)
    if kwargs['task_type']=='CPU':kwargs['thread_count']=min(4,os.cpu_count() or 2)
    clf=CatBoostClassifier(**kwargs);clf.fit(xx,yy)
    return clf

def cat_prob(clf,rid):
    raw=clf.predict_proba(SEGMENT_CACHE[rid]).mean(axis=0)
    p=np.zeros(8,dtype=np.float32);p[np.asarray(clf.classes_,dtype=int)]=raw
    p/=p.sum()
    assert p.shape==(8,) and np.isfinite(p).all() and abs(float(p.sum())-1)<1e-5
    return p

OOF_PATH=WEIGHTS/'train_gesture_day_oof.npz'
PROV_PATH=WEIGHTS/'train_proxy_provenance.json'
valid_cache=False
if REUSE_CHECKPOINTS and OOF_PATH.exists() and PROV_PATH.exists():
    manifest=json.loads(PROV_PATH.read_text())
    if manifest.get('training_signature')==TRAIN_SIGNATURE and set(manifest.get('records',{}))==set(TRAIN_IDS):
        with np.load(OOF_PATH) as z: TRAIN_PROXY={rid:z[rid].copy() for rid in z.files}
        valid_cache=set(TRAIN_PROXY)==set(TRAIN_IDS) and all(v.shape==(8,) and np.isfinite(v).all() and np.isclose(v.sum(),1) for v in TRAIN_PROXY.values())
if not valid_cache:
    TRAIN_PROXY={};provenance={}
    for held_day in DAYS:
        train_ids=[r for r in TRAIN_IDS if records[r]['day']!=held_day]
        val_ids=[r for r in TRAIN_IDS if records[r]['day']==held_day]
        assert not set(train_ids)&set(val_ids)
        clf=make_catboost(train_ids,seed=42)
        for rid in val_ids:
            TRAIN_PROXY[rid]=cat_prob(clf,rid)
            provenance[rid]={'heldout_day':held_day,'fit_ids':train_ids,'fit_days':sorted({records[r]['day'] for r in train_ids})}
        print('Day OOF:',held_day,'Train:',len(train_ids),'OOF:',len(val_ids),flush=True)
        del clf;gc.collect()
    assert set(TRAIN_PROXY)==set(TRAIN_IDS)
    assert all(rid not in p['fit_ids'] and records[rid]['day'] not in p['fit_days'] for rid,p in provenance.items())
    np.savez_compressed(OOF_PATH,**TRAIN_PROXY)
    PROV_PATH.write_text(json.dumps({'training_signature':TRAIN_SIGNATURE,'records':provenance},indent=2))
else:
    print('Reusing verified day-OOF training probabilities')

CAT_PATH=WEIGHTS/'gesture_catboost.cbm'; CAT_META=WEIGHTS/'gesture_catboost_meta.json'
if not (REUSE_CHECKPOINTS and CAT_PATH.exists() and CAT_META.exists() and json.loads(CAT_META.read_text()).get('training_signature')==TRAIN_SIGNATURE):
    full_clf=make_catboost(TRAIN_IDS,seed=42)
    full_clf.save_model(str(CAT_PATH))
    CAT_META.write_text(json.dumps({'training_signature':TRAIN_SIGNATURE,'fit_ids':TRAIN_IDS,'trees':PROXY_TREES},indent=2))
    del full_clf;gc.collect()
print('CatBoost OOF and full-train classifier ready')

Day OOF: day_14fb2381 Train: 72 OOF: 8
Day OOF: day_2e435bb4 Train: 72 OOF: 8
Day OOF: day_6133f7ce Train: 72 OOF: 8
Day OOF: day_c7670e91 Train: 64 OOF: 16
Day OOF: day_cda5287b Train: 72 OOF: 8
Day OOF: day_efee22d1 Train: 56 OOF: 24
Day OOF: day_fad41c15 Train: 72 OOF: 8
CatBoost OOF and full-train classifier ready


## 6. Глобальные классификаторы жеста — только train

In [13]:
GLOBAL_PATH=WEIGHTS/'whole_record_classifiers.joblib';GLOBAL_META=WEIGHTS/'whole_record_classifiers_meta.json'
if not (REUSE_CHECKPOINTS and GLOBAL_PATH.exists() and GLOBAL_META.exists() and json.loads(GLOBAL_META.read_text()).get('training_signature')==TRAIN_SIGNATURE):
    X=np.stack([whole_record_features(np.load(DATA_DIR/'train/signals'/f'{rid}.npy',allow_pickle=False)) for rid in TRAIN_IDS])
    y=np.asarray([ACTIONS.index(records[r]['action']) for r in TRAIN_IDS],dtype=int)
    classifiers=[make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=3000)),
                 make_pipeline(StandardScaler(),SVC(C=10,probability=True,random_state=42)),
                 make_pipeline(StandardScaler(),LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'))]
    for clf in classifiers:
        clf.fit(X,y);print('Trained whole-record:',type(clf[-1]).__name__,flush=True)
    joblib.dump(classifiers,GLOBAL_PATH)
    GLOBAL_META.write_text(json.dumps({'training_signature':TRAIN_SIGNATURE,'fit_ids':TRAIN_IDS},indent=2))
    del X,y,classifiers;gc.collect()
else:
    print('Reusing verified full-train whole-record classifiers')

Trained whole-record: LogisticRegression
Trained whole-record: SVC
Trained whole-record: LinearDiscriminantAnalysis


In [14]:
BASE_CACHE={rid:offline_features(records[rid]['x']) for rid in TRAIN_IDS}
SPECTRAL_CACHE={rid:spectral_offline_features(records[rid]['x']) for rid in TRAIN_IDS}
assert all(x[1].shape[1]==216 for x in BASE_CACHE.values())
assert all(x[1].shape[1]==360 for x in SPECTRAL_CACHE.values())

def baseline_stats(ids):
    sums=np.zeros(216,dtype=np.float64);sums2=np.zeros(216,dtype=np.float64);n=0
    for rid in ids:
        v=BASE_CACHE[rid][1].astype(np.float64)
        sums+=v.sum(0);sums2+=np.square(v).sum(0);n+=len(v)
    mu=sums/n;sd=np.sqrt(np.maximum(sums2/n-mu**2,1e-8))
    return mu.astype(np.float32),sd.astype(np.float32)

def corrected_stats(cache,ids):
    allft=np.concatenate([cache[r][1] for r in ids]).astype(np.float64)
    return allft.mean(0).astype(np.float32), np.maximum(allft.std(0),1e-12).astype(np.float32)

BASE_STATS=baseline_stats(TRAIN_IDS)
TEACHER_STATS=corrected_stats(BASE_CACHE,TRAIN_IDS)
SPECTRAL_STATS=corrected_stats(SPECTRAL_CACHE,TRAIN_IDS)
SPECTRAL_PHYSICAL=np.concatenate([SPECTRAL_CACHE[r][1] for r in TRAIN_IDS]).astype(np.float64)
ASINH_TAU=np.maximum(np.median(np.abs(SPECTRAL_PHYSICAL),axis=0),1e-12).astype(np.float32)
ALL_ASINH=np.arcsinh(SPECTRAL_PHYSICAL/ASINH_TAU)
ASINH_STATS=(ALL_ASINH.mean(0).astype(np.float32),np.maximum(ALL_ASINH.std(0),1e-12).astype(np.float32))
del SPECTRAL_PHYSICAL,ALL_ASINH;gc.collect()
ASINH_CACHE={rid:(grid,np.arcsinh(ft/ASINH_TAU).astype(np.float32)) for rid,(grid,ft) in SPECTRAL_CACHE.items()}
assert len(ASINH_TAU)==360

FINGERS=('Index','Middle','Little','Ring','Thumb')
def finger_weights(action):
    if action in ('Fist','Open','Close'): return np.full(20,.05,dtype=np.float32)
    w=np.full(20,.125/4,dtype=np.float32)
    w[4*FINGERS.index(action):4*FINGERS.index(action)+4]=.5/4
    return w

def loss_weighted(pred,y,mask,w):
    pred=pred[:,:,CONTEXT:CONTEXT+CORE]
    assert pred.shape==y.shape
    err=((pred-y).abs()*w[:,:,None]).sum(1)
    return (err*mask).sum()/mask.sum().clamp_min(1)

def prepared_series(rid,cache,stats,prob):
    grid,ft=cache[rid]
    mu,sd=stats
    x=np.clip((ft-mu)/sd,-10,10).astype(np.float32)
    if prob is not None:
        p=np.asarray(prob,dtype=np.float32)
        assert p.shape==(8,)
        x=np.concatenate([x,np.broadcast_to((p-.125)*4,(len(x),8))],axis=1)
    return grid,np.ascontiguousarray(x.T,dtype=np.float32)

class TrainPatches(Dataset):
    def __init__(self,ids,cache,stats,probs,kind,aug_gain=0.,balance=False):
        self.kind=kind;self.aug_gain=aug_gain;self.balance=balance
        self.nfeat=len(stats[0]);self.mu=torch.as_tensor(stats[0])[:,None];self.sd=torch.as_tensor(stats[1])[:,None]
        self.offset=torch.as_tensor(stats[0]/stats[1])[:,None]
        self.series={rid:prepared_series(rid,cache,stats,probs[rid]) for rid in ids}
        self.indices=[]
        for rid in ids:
            grid,_=self.series[rid]
            for start in range(0,len(grid),CORE):
                if np.any(grid[start:min(len(grid),start+CORE)]>=CONTEXT_SAMPLES):self.indices.append((rid,start))
        days={records[r]['day'] for r in ids}
        count={d:sum(records[r]['day']==d for r in ids) for d in days}
        self.day_weights={r:len(ids)/len(days)/count[records[r]['day']] for r in ids}
        assert self.nfeat in (216,360) and len(self.indices)>0
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        rid,start=self.indices[i];grid,feat=self.series[rid]
        xx=pad_and_slice(feat,start-CONTEXT,CORE+2*CONTEXT)
        p=np.arange(start,start+CORE);valid=np.clip(p,0,len(grid)-1);idx=grid[valid]
        y=records[rid]['y'][idx].T.astype(np.float32)
        mask=((p<len(grid))&(idx>=CONTEXT_SAMPLES)).astype(np.float32)
        x=torch.from_numpy(xx)
        w=torch.from_numpy(finger_weights(records[rid]['action']))
        if self.kind!='baseline':
            action=ACTIONS.index(records[rid]['action'])
            confidence=np.random.uniform(.3,1.)
            proxy=np.full(8,(1-confidence)/8,dtype=np.float32);proxy[action]+=confidence
            if np.random.random()<.1:proxy[:]=.125
            x[-8:]=torch.as_tensor((proxy-.125)*4)[:,None]
            if self.kind=='asinh':
                gain=torch.exp(torch.randn(6)*.2).repeat(60)[:,None]
                u=x[:360]*self.sd+self.mu
                x[:360]=((torch.asinh(torch.sinh(u)*gain)-self.mu)/self.sd).clamp(-10,10)
            elif self.aug_gain:
                gain=torch.exp(torch.randn(6)*self.aug_gain).repeat(self.nfeat//6)[:,None]
                x[:self.nfeat]=(x[:self.nfeat]*gain+(gain-1)*self.offset).clamp(-10,10)
            if self.balance:w=w*self.day_weights[rid]
        return x,torch.from_numpy(y),torch.from_numpy(mask),w

TRAIN_ONEHOT={r:np.eye(8,dtype=np.float32)[ACTIONS.index(records[r]['action'])] for r in TRAIN_IDS}
print('Train patches:',len(TrainPatches(TRAIN_IDS,BASE_CACHE,BASE_STATS,TRAIN_PROXY,'baseline')))

Train patches: 720


## 8. Обучение трёх базовых моделей (по 40 эпох)

In [15]:
ARCHITECTURES={
 'tcn': (lambda: RegressionTCN(224)),
 'bigru': (lambda: BiGRURegressor(224)),
 'hybrid': (lambda: HybridTCNBiGRU(224)),
 'teacher': (lambda: HybridTCNBiGRU(224)),
 'teacher_spectral': (lambda: HybridTCNBiGRU(368)),
 'teacher_spectral_seed27183': (lambda: HybridTCNBiGRU(368)),
 'teacher_group_gain_day': (lambda: GroupHybrid(224)),
 'teacher_asinh': (lambda: HybridTCNBiGRU(368)),
}

def train_model(name,cache,stats,probs,kind,epochs,seed,weight_decay,gain=0.,balance=False,asinh_tau=None):
    path=WEIGHTS/(name+'.pt')
    cfg={'signature':TRAIN_SIGNATURE,'name':name,'epochs':epochs,'seed':seed,'kind':kind,
         'weight_decay':weight_decay,'gain':gain,'balance':balance}
    if REUSE_CHECKPOINTS and path.exists():
        saved=torch.load(path,map_location='cpu',weights_only=False)
        if saved.get('train_config')==cfg and set(saved.get('train',[]))==set(TRAIN_IDS):
            assert len(saved['stats'][0])==len(stats[0]);print('REUSE VERIFIED',name,flush=True);return
    seed_all(seed)
    ds=TrainPatches(TRAIN_IDS,cache,stats,probs,kind,aug_gain=gain,balance=balance)
    loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=DEVICE.type=='cuda')
    net=ARCHITECTURES[name]().to(DEVICE)
    ema=copy.deepcopy(net).eval() if kind!='baseline' else None
    opt=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=weight_decay)
    scheduler=(torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs,eta_min=3e-5) if ema is not None else None)
    history=[]
    for epoch in range(1,epochs+1):
        net.train();losses=[]
        for x,y,mask,w in loader:
            x,y,mask,w=[t.to(DEVICE,non_blocking=True) for t in (x,y,mask,w)]
            opt.zero_grad(set_to_none=True)
            loss=loss_weighted(net(x),y,mask,w)
            if not torch.isfinite(loss):raise ValueError(f'Nonfinite loss {name} epoch {epoch}')
            loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1);opt.step()
            if ema is not None:
                with torch.no_grad():
                    for q,p in zip(ema.parameters(),net.parameters()):q.lerp_(p,.05)
                    for q,p in zip(ema.buffers(),net.buffers()):q.copy_(p)
            losses.append(float(loss.detach().cpu()))
        if scheduler is not None:scheduler.step()
        history.append({'epoch':epoch,'train_weighted_l1':float(np.mean(losses))})
        if epoch==1 or epoch==epochs or epoch%10==0:
            print(f'{name}: {epoch:2}/{epochs} weighted_L1={history[-1]["train_weighted_l1"]:.6f}',flush=True)
    chosen=ema if ema is not None else net
    saved={'state_dict':{k:v.detach().cpu() for k,v in chosen.state_dict().items()},'stats':stats,'train':TRAIN_IDS,
           'epochs':epochs,'seed':seed,'train_config':cfg,'data_fingerprint':DATA_FINGERPRINT,'history':history}
    if asinh_tau is not None:saved['tau']=asinh_tau
    torch.save(saved,path)
    print('SAVED',path.name,'parameters',sum(p.numel() for p in chosen.parameters()),flush=True)
    del net,ema,loader,ds,chosen;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()

for name in ('tcn','bigru','hybrid'):
    train_model(name,BASE_CACHE,BASE_STATS,TRAIN_PROXY,kind='baseline',epochs=BASE_EPOCHS,
                seed=42,weight_decay=.001)

tcn:  1/30 weighted_L1=0.411856
tcn: 10/30 weighted_L1=0.132286
tcn: 20/30 weighted_L1=0.107418
tcn: 30/30 weighted_L1=0.096725
SAVED tcn.pt parameters 140500
bigru:  1/30 weighted_L1=0.287622
bigru: 10/30 weighted_L1=0.104576
bigru: 20/30 weighted_L1=0.083648
bigru: 30/30 weighted_L1=0.071780
SAVED bigru.pt parameters 214292
hybrid:  1/30 weighted_L1=0.268960
hybrid: 10/30 weighted_L1=0.113897
hybrid: 20/30 weighted_L1=0.094592
hybrid: 30/30 weighted_L1=0.086009
SAVED hybrid.pt parameters 284340


## 9. Обучение пяти teacher-моделей (по 60 эпох)

In [16]:
train_model('teacher',BASE_CACHE,TEACHER_STATS,TRAIN_ONEHOT,kind='teacher',epochs=TEACHER_EPOCHS,
            seed=16180,weight_decay=.003)
train_model('teacher_spectral',SPECTRAL_CACHE,SPECTRAL_STATS,TRAIN_ONEHOT,kind='teacher',epochs=TEACHER_EPOCHS,
            seed=16180,weight_decay=.003,gain=.2)
train_model('teacher_spectral_seed27183',SPECTRAL_CACHE,SPECTRAL_STATS,TRAIN_ONEHOT,kind='teacher',epochs=TEACHER_EPOCHS,
            seed=27183,weight_decay=.003,gain=.2)
train_model('teacher_group_gain_day',BASE_CACHE,TEACHER_STATS,TRAIN_ONEHOT,kind='teacher',epochs=TEACHER_EPOCHS,
            seed=16180,weight_decay=.003,gain=.2,balance=True)
train_model('teacher_asinh',ASINH_CACHE,ASINH_STATS,TRAIN_ONEHOT,kind='asinh',epochs=TEACHER_EPOCHS,
            seed=16180,weight_decay=.003,asinh_tau=ASINH_TAU)
assert all((WEIGHTS/(n+'.pt')).exists() for n in ARCHITECTURES)

teacher:  1/60 weighted_L1=0.279118
teacher: 10/60 weighted_L1=0.109293
teacher: 20/60 weighted_L1=0.096931
teacher: 30/60 weighted_L1=0.083474
teacher: 40/60 weighted_L1=0.076648
teacher: 50/60 weighted_L1=0.071594
teacher: 60/60 weighted_L1=0.070478
SAVED teacher.pt parameters 284340
teacher_spectral:  1/60 weighted_L1=0.289418
teacher_spectral: 10/60 weighted_L1=0.121558
teacher_spectral: 20/60 weighted_L1=0.097332
teacher_spectral: 30/60 weighted_L1=0.087929
teacher_spectral: 40/60 weighted_L1=0.083134
teacher_spectral: 50/60 weighted_L1=0.077595
teacher_spectral: 60/60 weighted_L1=0.076251
SAVED teacher_spectral.pt parameters 293556
teacher_spectral_seed27183:  1/60 weighted_L1=0.315270
teacher_spectral_seed27183: 10/60 weighted_L1=0.120013
teacher_spectral_seed27183: 20/60 weighted_L1=0.101244
teacher_spectral_seed27183: 30/60 weighted_L1=0.090451
teacher_spectral_seed27183: 40/60 weighted_L1=0.082467
teacher_spectral_seed27183: 50/60 weighted_L1=0.078778
teacher_spectral_seed271

## 10. Манифест, checksum и проверка V4

In [17]:
import importlib.metadata as md
required=[n+'.pt' for n in ARCHITECTURES]+['gesture_catboost.cbm','whole_record_classifiers.joblib']
assert all((WEIGHTS/n).is_file() for n in required)
for n in ARCHITECTURES:
    saved=torch.load(WEIGHTS/(n+'.pt'),map_location='cpu',weights_only=False)
    assert saved['data_fingerprint']==DATA_FINGERPRINT
    assert saved['train_config']['signature']==TRAIN_SIGNATURE
    assert saved['epochs']==(BASE_EPOCHS if n in ('tcn','bigru','hybrid') else TEACHER_EPOCHS)
    assert set(saved['train'])==set(TRAIN_IDS)

checksums={n:hashlib.sha256((WEIGHTS/n).read_bytes()).hexdigest() for n in required}
versions={m:md.version(m) for m in ('numpy','scipy','torch','catboost','scikit-learn','joblib')}
manifest={'candidate':'v4_asinh_fixed_epochs','status':'FULLTRAIN_REFIT_UNSCORED','train_count':len(TRAIN_IDS),
          'train_ids':TRAIN_IDS,'data_fingerprint':DATA_FINGERPRINT,'train_signature':TRAIN_SIGNATURE,
          'epochs':{'tcn':BASE_EPOCHS,'bigru':BASE_EPOCHS,'hybrid':BASE_EPOCHS,
                    **{n:TEACHER_EPOCHS for n in ARCHITECTURES if n.startswith('teacher')}},
          'mixing':'V4 original: 1/2 equal3+teacher, 1/4 spectral group, 1/4 asinh as implemented in AIRIModel.predict',
          'test_selection':TEST_VERSION,'sha256':checksums,'versions':versions,
          'validation_reference_only':{'historical_cv4':0.09907291067808169,'historical_lodo':0.10104469175632928,
              'warning':'Historical fixed-15/16/17 base epochs. This full refit uses 40, cannot claim same CV.'}}
(WEIGHTS/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
model=AIRIModel(weights=WEIGHTS,device=DEVICE)
probe=model.predict(np.zeros((1001,6),dtype=np.float32))
assert probe.shape==(1001,20) and np.isfinite(probe).all()
print('V4 complete ensemble: 8/8 loaded; checksum PASS; minimum input PASS.')

V4 complete ensemble: 8/8 loaded; checksum PASS; minimum input PASS.


## 11. Инференс и сабмит — строго выбранная фаза

In [20]:
if RUN_FINAL_INFERENCE:
    assert TEST_VERSION in ('v1','v2')
    if TEST_VERSION=='v2':
        assert TEMPLATE.name=='phase2_sample_submission.csv', 'Вместо V2 подсунут V1-шаблон'
        assert len(template_record_ids)==32, 'Это не реальный V2 test с 32 записями'
        assert set(p.stem for p in TEST_SIGNALS.glob('rec_*.npy'))==template_record_ids
    else:
        assert TEMPLATE.name=='phase1_sample_submission.csv'
    output=OUT/('submission.csv' if TEST_VERSION=='v2' else 'phase1_format_submission.csv')
    assert output.parent==OUT and not output.is_relative_to(DATA_DIR)
    result=write_submission(model,TEST_SIGNALS,TEMPLATE,output)
    assert result['recordings']==len(template_record_ids) and result['rows']==template_row_count
    with output.open(encoding='utf-8',newline='') as f:
        csv_reader=csv.DictReader(f)
        assert csv_reader.fieldnames==REQUIRED_COLUMNS
        check_rows=0
        for row in csv_reader:
            assert all(np.isfinite(float(row[f'angle_{j:02d}'])) for j in range(20))
            check_rows+=1
    assert check_rows==template_row_count
    if TEST_VERSION=='v2' and Path('/kaggle/working').is_dir():
        import shutil
        canonical=Path('/kaggle/working/submission.csv')
        if canonical.resolve()!=output.resolve():
            temp_copy=canonical.with_suffix('.csv.partial')
            shutil.copyfile(output,temp_copy);os.replace(temp_copy,canonical)
        print('KAGGLE FINAL SUBMISSION:',canonical,flush=True)
    print('SUBMISSION GENERATED:',output,'rows:',check_rows,'test:',TEST_VERSION,flush=True)

KAGGLE FINAL SUBMISSION: /kaggle/working/submission.csv
SUBMISSION GENERATED: /kaggle/working/airi_v4_fulltrain/submission.csv rows: 624032 test: v2
